In [ ]:

import os
import pickle

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA

# Paths and settings
BASE = ""
EMB_DIR = os.path.join(BASE, "meanpooling")
OUT_DIR = os.path.join(BASE, "PCA")

MODALITIES = ["http", "file", "email"]
USERS = [ ]

EMB_DIM_EXPECTED = 384
N_COMPONENTS = 32           #PCA dimension
EXPECTED_KEYS = {"user", "modality", "dates", "embeddings"}

# Load mean-pooled embeddings for all users
emb_store = {}

for modality in MODALITIES:
    input_dir = os.path.join(EMB_DIR, modality)
    os.makedirs(os.path.join(OUT_DIR, modality), exist_ok=True)

    for user in USERS:
        path = os.path.join(input_dir, f"{modality}_{user}_meanpooled.pkl")
        if not os.path.isfile(path):
            raise FileNotFoundError(f"File not found: {path}")

        with open(path, "rb") as f:
            obj = pickle.load(f)

        if not isinstance(obj, dict) or not EXPECTED_KEYS.issubset(obj):
            raise ValueError(f"[{modality}/{user}] Invalid PKL structure")
        if obj["user"] != user or obj["modality"] != modality:
            raise ValueError(f"[{modality}/{user}] User or modality mismatch")

        dates = [str(pd.to_datetime(d).date()) for d in obj["dates"]]
        embeddings = np.asarray(obj["embeddings"], dtype=np.float32)

        if embeddings.ndim != 2 or embeddings.shape != (len(dates), EMB_DIM_EXPECTED):
            raise ValueError(f"[{modality}/{user}] Invalid embedding shape")
        if not np.isfinite(embeddings).all():
            raise ValueError(f"[{modality}/{user}] Embeddings contain NaN or Inf")
        if len(set(dates)) != len(dates):
            raise ValueError(f"[{modality}/{user}] Duplicate dates")

        if dates != sorted(dates):
            order = np.argsort(dates)
            dates = [dates[i] for i in order]
            embeddings = embeddings[order]

        emb_store[(modality, user)] = {
            "dates": dates,
            "embeddings": embeddings,
            "src": path,
        }

# Fit PCA separately for each modality and save each user's transformed embeddings
for modality in MODALITIES:
    X = np.concatenate(
        [emb_store[(modality, user)]["embeddings"] for user in USERS],
        axis=0,
    ).astype(np.float64)

    if X.shape[0] < N_COMPONENTS:
        raise ValueError(f"[{modality}] Insufficient samples for PCA")

    pca = PCA(n_components=N_COMPONENTS, svd_solver="full", random_state=42)
    pca.fit(X)
    explained_variance = float(np.cumsum(pca.explained_variance_ratio_)[-1])

    for user in USERS:
        record = emb_store[(modality, user)]
        transformed = pca.transform(
            record["embeddings"].astype(np.float64)
        ).astype(np.float32)

        if transformed.shape != (len(record["dates"]), N_COMPONENTS):
            raise ValueError(f"[{modality}/{user}] Invalid PCA output shape")
        if not np.isfinite(transformed).all():
            raise ValueError(f"[{modality}/{user}] PCA output contains NaN or Inf")

        output_path = os.path.join(
            OUT_DIR, modality, f"{modality}_{user}_pca{N_COMPONENTS}.pt"
        )
        torch.save({
            "user": user,
            "modality": modality,
            "dates": record["dates"],
            "embeddings": torch.from_numpy(np.ascontiguousarray(transformed)),
            "input_dim": EMB_DIM_EXPECTED,
            "n_components": N_COMPONENTS,
            "source_pkl": record["src"],
            "explained_variance_ratio_sum": explained_variance,
        }, output_path)

    print(f"[{modality.upper()}] PCA saved: {len(USERS)} files, "
          f"explained variance: {explained_variance:.2%}")

print(f"PCA completed: {OUT_DIR}")
